# Hybrid VQC Training on BCCC-CIC-IDS-2017 (Resumable)

This notebook trains a **Hybrid Variational Quantum Classifier (VQC)** on the BCCC-CIC-IDS-2017 dataset.
It aligns with the updated QNN training workflow, including **checkpointing**, **advanced metrics**, and **resume capability**.

In [6]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import pennylane as qml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## 1. Configuration & Data Loading

In [7]:
# Configuration
DATASET_PATH = r'Datasets/processed_data.csv'
BATCH_SIZE = 32
EPOCHS = 52
LEARNING_RATE = 0.01
N_QUBITS = 4        # VQC Width
N_LAYERS = 2        # VQC Depth
SEED = 42

# Checkpoint Configuration
CHECKPOINT_DIR = 'checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, 'checkpoint_ids2017_vqc.pth')

np.random.seed(SEED)
torch.manual_seed(SEED)

In [8]:
print("Loading dataset...")
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}. Please run preprocess_data.py first.")

df = pd.read_csv(DATASET_PATH)
print(f"Total samples: {len(df)}")

# Check for label column
if 'label' not in df.columns:
    raise ValueError("Column 'label' not found in dataset.")

# Separate Features and Target
X = df.drop(columns=['label']).values
y = df['label'].values

# Determine number of classes
num_classes = len(np.unique(y))
print(f"Number of classes: {num_classes}")

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Adjust for Binary vs Multiclass
if num_classes == 2:
    y_train_tensor = y_train_tensor.float().unsqueeze(1)
    y_test_tensor = y_test_tensor.float().unsqueeze(1)
    criterion = nn.BCEWithLogitsLoss()
    output_dim = 1
    print("Binary Classification detected.")
else:
    criterion = nn.CrossEntropyLoss()
    output_dim = num_classes
    print("Multiclass Classification detected.")

# Create DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")
print(f"Input Feature Dimension: {X_train.shape[1]}")

Loading dataset...
Total samples: 840549
Number of classes: 14
Multiclass Classification detected.
Training samples: 672439
Testing samples: 168110
Input Feature Dimension: 116


## 2. Hybrid VQC Architecture

In [ ]:
# Quantum Device
dev = qml.device("default.qubit", wires=N_QUBITS)

@qml.qnode(dev, interface="torch")
def vqc_circuit(inputs, weights):
    # Feature Map: Angle Embedding (more efficient for VQC)
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS))
    
    # Ansatz: Strongly Entangling Layers
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    
    # Measurement: Expectation values
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

class HybridVQC(nn.Module):
    def __init__(self, input_dim, n_qubits, n_layers, output_dim):
        super(HybridVQC, self).__init__()
        self.n_qubits = n_qubits
        
        # Classical Encoder: Project High-Dim -> Qubits
        self.cl_layer1 = nn.Linear(input_dim, 64)
        self.cl_layer2 = nn.Linear(64, n_qubits)
        
        # Quantum Layer (TorchLayer handles batchin
        # g automatically)
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.vqc = qml.qnn.TorchLayer(vqc_circuit, weight_shapes)
        
        # Classical Decoder: Qubits -> Classes
        self.cl_layer3 = nn.Linear(n_qubits, output_dim)
        
    def forward(self, x):
        # Classical encoding
        x = torch.relu(self.cl_layer1(x))
        x = self.cl_layer2(x)
        
        # Squash to [-pi, pi] for Angle Embedding
        x = torch.tanh(x) * np.pi 
        
        # VQC
        x = self.vqc(x)
        
        # Final Classification
        x = self.cl_layer3(x)
        return x

model = HybridVQC(X_train.shape[1], N_QUBITS, N_LAYERS, output_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)

HybridVQC(
  (cl_layer1): Linear(in_features=116, out_features=64, bias=True)
  (cl_layer2): Linear(in_features=64, out_features=4, bias=True)
  (vqc): <Quantum Torch Layer: func=vqc_circuit>
  (cl_layer3): Linear(in_features=4, out_features=14, bias=True)
)


## 3. Training Loop with Resume Capability

In [28]:
def evaluate(model, loader, is_binary):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            if is_binary:
                preds = torch.sigmoid(logits) > 0.5
            else:
                preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    
    average_type = 'binary' if is_binary else 'weighted'
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average=average_type, zero_division=0)
    rec = recall_score(all_labels, all_preds, average=average_type, zero_division=0)
    f1 = f1_score(all_labels, all_preds, average=average_type, zero_division=0)
    return acc, prec, rec, f1

# Check for checkpoint
start_epoch = 0
if os.path.exists(CHECKPOINT_PATH):
    print(f"Loading checkpoint from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"Resuming training from epoch {start_epoch+1}")
else:
    print("No checkpoint found. Starting training from scratch.")

print("Starting training...")
is_binary = (num_classes == 2)

if start_epoch < EPOCHS:
    for ep in range(start_epoch, EPOCHS):
        model.train()
        total_loss = 0
        for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {ep+1}/{EPOCHS}"):
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        acc, prec, rec, f1 = evaluate(model, test_loader, is_binary)
        
        print(f"Epoch {ep+1}: Loss = {avg_loss:.4f}")
        print(f"  Test Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")
        
        # Save Checkpoint
        checkpoint = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f"Checkpoint saved for Epoch {ep+1}")
else:
    print("Training already completed!")

# Save Final Model
save_dir = 'trained_model'
os.makedirs(save_dir, exist_ok=True)
model_path = os.path.join(save_dir, 'hybrid_vqc_ids2017.pth')
torch.save(model.state_dict(), model_path)
print(f"Final Model saved to {model_path}")

Loading checkpoint from checkpoints\checkpoint_ids2017_vqc.pth...
Resuming training from epoch 51
Starting training...


Epoch 51/52:   1%|          | 235/21014 [00:03<05:40, 61.08it/s]


KeyboardInterrupt: 